### Data merge and feature engineering


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from tqdm import tqdm
import pymorphy3
from collections import Counter

import os
from pathlib import Path
import json

In [31]:
df_weather = pd.read_csv("../data/weather_processed.csv")
df_war_events = pd.read_csv("../data/war_events_processed.csv")
df_isw = pd.read_csv("../data/isw_processed.csv")
df_isw_matrix = pd.read_csv("../data/matrix_isw.csv")
df_tg = pd.read_csv("../data/telegram_processed.csv")

In [3]:
df_weather['datetime_hour'] = pd.to_datetime(df_weather['datetime_hour'], errors="coerce")
df_war_events['datetime_hour'] = pd.to_datetime(df_war_events['datetime_hour'], errors="coerce")

In [4]:
df_final = pd.merge(
    df_weather, 
    df_war_events[['datetime_hour', 'region_id', 'alarm_active', 'alarm_minutes_in_hour']], 
    on=['datetime_hour', 'region_id'], 
    how='left'
)
df_final.sample(20)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,region_key,region_id,alarm_active,alarm_minutes_in_hour
293813,48.6264,22.2851,Uzhgorod,Europe/Uzhgorod,2023-07-19,20.9,15.7,73.9,44.992,100.0,...,True,False,False,True,False,False,Закарпатська,7,0,0.000000
483813,48.2924,25.9355,Chernivtsi,Europe/Kiev,2024-06-12,18.7,12.6,69.3,0.347,100.0,...,True,False,False,True,False,False,Чернівецька,24,0,0.000000
418155,48.0020,37.8145,Donetsk,Europe/Kiev,2024-02-19,-0.9,-5.9,71.1,0.300,100.0,...,False,True,False,True,False,False,Донецька,5,1,60.000000
202226,48.4753,35.0160,Dnipro,Europe/Kiev,2023-02-10,-3.5,-7.6,73.9,0.000,0.0,...,False,False,False,True,False,False,Дніпропетровська,4,0,0.000000
425549,48.6264,22.2851,Uzhgorod,Europe/Uzhgorod,2024-03-03,9.2,5.8,80.6,0.000,0.0,...,False,False,False,True,False,False,Закарпатська,7,0,0.000000
110354,48.4753,35.0160,Dnipro,Europe/Kiev,2022-09-03,17.5,4.6,43.5,0.436,100.0,...,True,False,False,True,False,False,Дніпропетровська,4,0,0.000000
565965,48.2924,25.9355,Chernivtsi,Europe/Kiev,2024-11-02,9.6,3.3,65.4,0.652,100.0,...,True,False,False,True,False,False,Чернівецька,24,0,0.000000
612007,48.9226,24.7147,Ivano-Frankivsk,Europe/Kiev,2025-01-21,0.2,-0.7,93.8,0.000,0.0,...,False,False,False,True,False,False,Івано-Франківська,9,0,0.000000
362020,50.2536,28.6654,Zhytomyr,Europe/Kiev,2023-11-14,4.9,2.9,87.3,6.000,100.0,...,True,False,False,True,False,False,Житомирська,6,0,0.000000
44568,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-05-12,17.6,5.4,46.7,0.000,0.0,...,False,False,False,True,False,False,Вінницька,2,0,0.000000


In [5]:
df_final = df_final.sort_values(['region_id', 'datetime_hour'])
df_final.head()

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,region_key,region_id,alarm_active,alarm_minutes_in_hour
0,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0
24,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0
48,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0
72,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0
96,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0


In [6]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 634752 entries, 0 to 634751
Data columns (total 56 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   city_latitude                  634752 non-null  float64       
 1   city_longitude                 634752 non-null  float64       
 2   city_name                      634752 non-null  str           
 3   city_timezone                  634752 non-null  str           
 4   day_datetime                   634752 non-null  str           
 5   day_temp                       634752 non-null  float64       
 6   day_dew                        634752 non-null  float64       
 7   day_humidity                   634752 non-null  float64       
 8   day_precip                     634752 non-null  float64       
 9   day_precipprob                 634752 non-null  float64       
 10  day_precipcover                634752 non-null  float64       
 11  day_snow        

In [7]:
df_final['alarms_in_last_24h'] = df_final.groupby('region_id')['alarm_active'].rolling(window=24, min_periods=1).sum().reset_index(level=0, drop=True)
df_final['alarm_in_last_hour'] = df_final.groupby('region_id')['alarm_active'].shift(1)
df_final['total_active_alarms'] = df_final.groupby('datetime_hour')['alarm_active'].transform('sum')
df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)

df_final.sample(10)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,hour_conditions_simple_Snow,region_key,region_id,alarm_active,alarm_minutes_in_hour,alarms_in_last_24h,alarm_in_last_hour,total_active_alarms,is_weekend,is_night
63696,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-06-14,17.5,8.9,58.9,0.1,100.0,...,False,Вінницька,2,0,0.000000,6.0,0.0,1,0,0
467426,48.4753,35.0160,Dnipro,Europe/Kiev,2024-05-15,11.6,0.9,48.6,0.0,0.0,...,False,Дніпропетровська,4,1,31.516667,15.0,1.0,10,0,0
60054,47.8289,35.1626,Zaporozhye,Europe/Zaporozhye,2022-06-08,23.4,10.6,47.8,0.0,0.0,...,False,Запорізька,8,1,21.150000,3.0,0.0,3,0,0
536304,49.2336,28.4486,Vinnytsia,Europe/Kiev,2024-09-12,19.9,16.9,83.7,13.0,100.0,...,False,Вінницька,2,1,60.000000,4.0,1.0,18,0,1
302249,50.0042,36.2358,Kharkiv,Europe/Kiev,2023-08-02,22.5,16.7,71.3,0.5,100.0,...,False,Харківська,20,0,0.000000,17.0,1.0,1,0,0
87800,50.4506,30.5243,Kyiv,Europe/Kiev,2022-07-26,21.7,12.8,59.8,0.0,0.0,...,False,Київська,10,0,0.000000,4.0,0.0,1,0,0
435541,49.5879,34.5517,Poltava,Europe/Kiev,2024-03-21,5.1,-0.3,69.8,0.0,0.0,...,False,Полтавська,16,1,56.250000,8.0,0.0,24,0,1
580452,46.4725,30.7371,Odesa,Europe/Kiev,2024-11-27,0.5,-1.8,84.1,0.0,0.0,...,False,Одеська,15,0,0.000000,4.0,0.0,1,0,0
474738,46.6401,32.6142,Kherson,Europe/Kiev,2024-05-28,20.8,11.6,58.0,0.3,100.0,...,False,Херсонська,21,1,35.133333,7.0,0.0,4,0,1
99904,49.5527,25.5889,Ternopil,Europe/Kiev,2022-08-16,20.3,18.0,87.1,4.0,100.0,...,False,Тернопільська,19,0,0.000000,2.0,0.0,1,0,0


In [8]:
neighbouring_regions = {
    1: [21],
    2: [6, 10, 11, 15, 22, 23, 24],
    3: [13, 17],
    4: [5, 8, 11, 14, 16, 20, 21],
    5: [4, 8, 12, 20],
    6: [2, 10, 17, 22],
    7: [9, 13],
    8: [4, 5, 21],
    9: [7, 13, 19, 24],
    10: [2, 6, 16, 23, 25],
    11: [2, 4, 14, 15, 16, 23],
    12: [5, 20],
    13: [3, 7, 9, 17, 19],
    14: [4, 11, 15, 21],
    15: [2, 11, 14],
    16: [4, 10, 11, 18, 20, 23, 25],
    17: [3, 6, 13, 19, 22],
    18: [16, 20, 25],
    19: [9, 13, 17, 22, 24],
    20: [4, 5, 12, 16, 18],
    21: [1, 4, 8, 14],
    22: [2, 6, 17, 19, 24],
    23: [2, 10, 11, 16],
    24: [2, 9, 19, 22],
    25: [10, 16, 18], 
    26: [10]
}

alarms_matrix = df_final.pivot_table(index='datetime_hour', columns='region_id', values='alarm_active', fill_value=0)

def get_neighbour_sum_safe(row):
    region_id = row['region_id']
    datetime = row['datetime_hour']
    
    if region_id in neighbouring_regions:
        neighbours = neighbouring_regions[region_id]
        
        if datetime in alarms_matrix.index:
            valid_neighbours = [n for n in neighbours if n in alarms_matrix.columns]

            if valid_neighbours:
                return alarms_matrix.loc[datetime, valid_neighbours].sum()
    return 0

df_final['neighbour_alarms'] = df_final.apply(get_neighbour_sum_safe, axis=1)

df_final.sample(15)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,region_key,region_id,alarm_active,alarm_minutes_in_hour,alarms_in_last_24h,alarm_in_last_hour,total_active_alarms,is_weekend,is_night,neighbour_alarms
158275,49.41680,26.97430,Khmelnytskyi,Europe/Kiev,2022-11-25,-0.5,-1.5,93.3,2.8,100.0,...,Хмельницька,22,0,0.000000,0.0,0.0,0,0,0,0.0
235433,50.00420,36.23580,Kharkiv,Europe/Kiev,2023-04-08,11.4,3.5,58.8,0.1,100.0,...,Харківська,20,0,0.000000,3.0,0.0,0,1,0,0.0
325757,48.62640,22.28510,Uzhgorod,Europe/Uzhgorod,2023-09-12,20.8,15.1,72.5,0.0,0.0,...,Закарпатська,7,0,0.000000,0.0,0.0,2,0,0,0.0
595257,48.50850,32.26560,Kropyvnytskyi,Europe/Kiev,2024-12-23,0.9,0.3,96.0,0.1,100.0,...,Кіровоградська,11,0,0.000000,6.0,0.0,2,0,0,0.0
147661,49.58790,34.55170,Poltava,Europe/Kiev,2022-11-07,2.5,-0.2,82.6,0.0,0.0,...,Полтавська,16,0,0.000000,5.0,0.0,0,0,0,0.0
52903,48.92260,24.71470,Ivano-Frankivsk,Europe/Kiev,2022-05-26,17.6,8.4,58.6,2.0,100.0,...,Івано-Франківська,9,1,60.000000,2.0,1.0,22,0,0,4.0
71915,46.97336,31.98522,Mykolaiv,Europe/Kiev,2022-06-28,26.5,14.0,49.9,0.0,0.0,...,Миколаївська,14,1,33.500000,8.0,1.0,9,0,0,2.0
571868,49.44070,32.06370,Cherkasy,Europe/Kiev,2024-11-12,3.5,-0.4,76.1,0.0,0.0,...,Черкаська,23,0,0.000000,13.0,0.0,5,0,0,1.0
67109,48.62636,22.28514,Uzhgorod,Europe/Uzhgorod,2022-06-20,23.7,11.5,49.6,0.0,0.0,...,Закарпатська,7,0,0.000000,0.0,0.0,2,0,0,0.0
385931,46.97340,31.98520,Mykolaiv,Europe/Kiev,2023-12-26,9.5,5.2,75.4,0.0,0.0,...,Миколаївська,14,1,26.333333,8.0,1.0,4,0,1,2.0


In [9]:
def calculate_hours_since_last(series):
    groups = series.cumsum()
    return series.groupby(groups).cumcount()

df_final['hours_since_last_alarm'] = df_final.groupby('region_id')['alarm_active'].transform(calculate_hours_since_last)

In [10]:
df_final['neighbour_alarms'] = df_final['neighbour_alarms'].astype(int)
df_final['alarms_in_last_24h'] = df_final['alarms_in_last_24h'].astype(int)
df_final['alarm_in_last_hour'] = df_final['alarm_in_last_hour'].fillna(0).astype(int)

In [11]:
df_final.sample(20)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,region_id,alarm_active,alarm_minutes_in_hour,alarms_in_last_24h,alarm_in_last_hour,total_active_alarms,is_weekend,is_night,neighbour_alarms,hours_since_last_alarm
621939,48.0020,37.8145,Donetsk,Europe/Kiev,2025-02-07,-5.7,-8.9,77.9,0.000,0.0,...,5,0,0.000000,17,0,2,0,0,1,2
209400,49.2336,28.4486,Vinnytsia,Europe/Kiev,2023-02-22,0.8,-2.0,81.9,1.000,100.0,...,2,0,0.000000,2,0,0,0,0,0,3
153761,50.0042,36.2358,Kharkiv,Europe/Kiev,2022-11-17,-2.1,-3.0,94.0,6.000,100.0,...,20,0,0.000000,9,0,0,0,0,0,4
74924,49.4407,32.0637,Cherkasy,Europe/Kiev,2022-07-04,24.1,13.8,56.5,0.000,0.0,...,23,0,0.000000,4,0,1,0,1,0,5
616102,51.4937,31.2944,Chernihiv,Europe/Kiev,2025-01-28,6.4,4.4,87.1,0.200,100.0,...,25,0,0.000000,10,0,3,0,0,1,3
603179,46.9734,31.9852,Mykolaiv,Europe/Kiev,2025-01-06,3.6,0.5,80.7,0.700,100.0,...,14,0,0.000000,10,0,11,0,1,3,2
187664,50.4506,30.5243,Kyiv,Europe/Kiev,2023-01-15,-1.3,-1.3,95.1,0.600,100.0,...,10,0,0.000000,0,0,0,1,0,0,26
246682,49.8444,24.0254,Lviv,Europe/Kiev,2023-04-28,6.8,0.1,65.8,0.600,100.0,...,13,0,0.000000,3,1,0,0,0,0,1
325833,48.5085,32.2656,Kropyvnytskyi,Europe/Kiev,2023-09-12,17.3,8.3,60.1,0.000,0.0,...,11,0,0.000000,1,0,4,0,0,1,8
396499,49.4168,26.9743,Khmelnytskyi,Europe/Kiev,2024-01-13,-3.4,-4.7,90.7,3.000,100.0,...,22,1,23.733333,3,1,24,1,0,5,0


In [12]:
df_isw_matrix.sample(10)

,date,isw_tfidf_activity,isw_tfidf_activity belarus,isw_tfidf_activity russianoccupied,isw_tfidf_additional,isw_tfidf_administration,isw_tfidf_advance,isw_tfidf_advance russian,isw_tfidf_advanced,isw_tfidf_air,...,isw_tfidf_western zaporizhia,isw_tfidf_westward,isw_tfidf_westward eastern,isw_tfidf_within,isw_tfidf_without,isw_tfidf_would,isw_tfidf_yar,isw_tfidf_zaporizhia,isw_tfidf_zaporizhia oblast,isw_tfidf_zelensky
891,2024-08-08,0.025827,0.009562,0.006837,0.000000,0.003409,0.036533,0.004094,0.026686,0.024986,...,0.003532,0.006158,0.006475,0.053077,0.005918,0.045704,0.016072,0.002655,0.002825,0.008902
935,2024-09-21,0.023584,0.010914,0.007804,0.023599,0.003891,0.029785,0.009347,0.054152,0.025351,...,0.004032,0.007029,0.007390,0.019132,0.003377,0.006956,0.041276,0.009091,0.009672,0.000000
1319,2025-10-14,0.022179,0.003910,0.000000,0.000000,0.004182,0.076831,0.025114,0.069114,0.017029,...,0.017333,0.003777,0.003972,0.006854,0.000000,0.000000,0.009858,0.019541,0.020791,0.010921
1313,2025-10-08,0.012845,0.003963,0.000000,0.000000,0.004238,0.064891,0.030544,0.077422,0.013808,...,0.004392,0.003829,0.004025,0.013894,0.011037,0.011366,0.014987,0.009902,0.007024,0.000000
592,2023-10-11,0.024408,0.012909,0.009230,0.004652,0.004602,0.028183,0.000000,0.064049,0.000000,...,0.028611,0.008314,0.008741,0.000000,0.003995,0.000000,0.000000,0.028672,0.030507,0.000000
372,2023-03-05,0.005016,0.000000,0.000000,0.006692,0.000000,0.055747,0.000000,0.011517,0.005392,...,0.000000,0.000000,0.000000,0.005425,0.005747,0.112433,0.000000,0.010311,0.005486,0.000000
1237,2025-07-24,0.020150,0.008289,0.000000,0.004481,0.008865,0.061078,0.021296,0.077114,0.018050,...,0.004593,0.008008,0.008419,0.029061,0.011543,0.075279,0.057471,0.006904,0.007346,0.057876
1200,2025-06-17,0.022595,0.009295,0.000000,0.010048,0.000000,0.064682,0.017909,0.099438,0.028336,...,0.005150,0.008979,0.009441,0.016293,0.004314,0.008885,0.064443,0.011612,0.004118,0.006490
995,2024-11-21,0.027347,0.011250,0.012066,0.004054,0.004010,0.027631,0.014451,0.104655,0.029397,...,0.004156,0.007245,0.007618,0.013147,0.010444,0.000000,0.056726,0.006247,0.003323,0.000000
473,2023-06-14,0.032502,0.013370,0.014340,0.000000,0.007149,0.016419,0.000000,0.018657,0.000000,...,0.022225,0.012916,0.013580,0.005859,0.006206,0.000000,0.000000,0.072384,0.047395,0.000000


In [13]:
df_isw_matrix["date"] = pd.to_datetime(df_isw_matrix["date"]) + pd.Timedelta(days=1)
df_isw_matrix = df_isw_matrix.rename(columns={'date': 'day_datetime'})

In [23]:
df_final['day_datetime'] = pd.to_datetime(df_final['day_datetime'])
df_isw_matrix['day_datetime'] = pd.to_datetime(df_isw_matrix['day_datetime'])
df_isw_matrix = df_isw_matrix.fillna(0)

In [24]:
df_final = df_final.merge(df_isw_matrix, on="day_datetime", how="left")

In [30]:
df_final.sample(20)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,isw_tfidf_western zaporizhia_y,isw_tfidf_westward_y,isw_tfidf_westward eastern_y,isw_tfidf_within_y,isw_tfidf_without_y,isw_tfidf_would_y,isw_tfidf_yar_y,isw_tfidf_zaporizhia_y,isw_tfidf_zaporizhia oblast_y,isw_tfidf_zelensky_y
281214,49.8444,24.0254,Lviv,Europe/Kiev,2023-12-20,4.6,-0.2,71.5,0.900,100.0,...,0.000000,0.009338,0.009818,0.004236,0.004487,0.004620,0.006093,0.000000,0.000000,0.060742
575282,48.2924,25.9355,Chernivtsi,Europe/Kiev,2024-03-27,7.5,1.7,68.1,0.000,0.0,...,0.006269,0.010930,0.011492,0.004958,0.010504,0.027041,0.007132,0.014136,0.015040,0.000000
145122,48.6264,22.2851,Uzhgorod,Europe/Uzhgorod,2023-07-27,16.2,9.9,67.6,0.063,100.0,...,0.020316,0.013283,0.009310,0.012051,0.012764,0.017525,0.000000,0.015269,0.016246,0.006400
539040,49.4407,32.0637,Cherkasy,Europe/Kiev,2023-02-16,-0.9,-4.9,75.0,0.600,100.0,...,0.000000,0.005525,0.005809,0.005013,0.010619,0.027338,0.000000,0.019055,0.015206,0.000000
277860,49.8444,24.0254,Lviv,Europe/Kiev,2023-08-02,19.6,15.4,78.1,12.000,100.0,...,0.035273,0.008786,0.009237,0.007971,0.012664,0.008694,0.000000,0.045447,0.040296,0.000000
566386,48.2924,25.9355,Chernivtsi,Europe/Kiev,2023-03-22,10.2,6.5,80.1,0.008,100.0,...,0.000000,0.005590,0.005877,0.030429,0.016115,0.044253,0.000000,0.019278,0.015384,0.000000
594127,51.4937,31.2944,Chernihiv,Europe/Kiev,2023-05-12,13.4,4.1,58.5,0.100,100.0,...,0.000000,0.010881,0.011440,0.004936,0.005228,0.016150,0.014198,0.000000,0.000000,0.031455
100437,48.0020,37.8145,Donetsk,Europe/Kiev,2024-07-09,28.9,12.5,38.6,0.000,0.0,...,0.010765,0.009384,0.009866,0.038311,0.004509,0.009286,0.061225,0.012136,0.008608,0.033911
205766,48.9226,24.7147,Ivano-Frankivsk,Europe/Kiev,2024-06-08,19.8,14.8,75.3,0.100,100.0,...,0.003427,0.005976,0.006283,0.024397,0.005743,0.038437,0.031191,0.015456,0.005482,0.082060
176757,47.8289,35.1626,Zaporozhye,Europe/Zaporozhye,2024-02-25,3.0,-2.6,67.1,0.000,0.0,...,0.009909,0.008639,0.009082,0.007837,0.008301,0.025645,0.000000,0.007448,0.007924,0.012487


In [29]:
df_final.isna().sum()

city_latitude                       0
city_longitude                      0
city_name                           0
city_timezone                       0
day_datetime                        0
                                 ... 
isw_tfidf_would_y                6336
isw_tfidf_yar_y                  6336
isw_tfidf_zaporizhia_y           6336
isw_tfidf_zaporizhia oblast_y    6336
isw_tfidf_zelensky_y             6336
Length: 1063, dtype: int64

In [20]:
df_nan_rows = df_final.loc[df_final.isnull().any(axis=1)]

In [22]:
df_nan_rows.sample(20)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,isw_tfidf_western zaporizhia,isw_tfidf_westward,isw_tfidf_westward eastern,isw_tfidf_within,isw_tfidf_without,isw_tfidf_would,isw_tfidf_yar,isw_tfidf_zaporizhia,isw_tfidf_zaporizhia oblast,isw_tfidf_zelensky
404427,50.9080,34.7972,Sumy,Europe/Kiev,2022-12-26,0.5,-0.3,94.8,5.651,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157423,48.6264,22.2851,Uzhgorod,Europe/Uzhgorod,2025-01-02,-0.6,-4.2,77.1,1.146,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95531,48.0020,37.8145,Donetsk,Europe/Kiev,2023-12-26,4.8,2.4,85.3,1.600,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
103674,48.0020,37.8145,Donetsk,Europe/Kiev,2024-11-29,0.1,-1.5,89.2,0.000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
263311,48.5085,32.2656,Kropyvnytskyi,Europe/Kiev,2025-01-02,3.9,-1.1,70.9,0.000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157263,48.6264,22.2851,Uzhgorod,Europe/Uzhgorod,2024-12-26,-0.3,-6.8,63.9,0.000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
606636,51.4937,31.2944,Chernihiv,Europe/Kiev,2024-11-29,0.1,-0.7,94.7,0.000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
616387,50.4506,30.5243,Kyiv,Europe/Kiev,2023-01-02,8.6,5.7,82.2,0.000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
475103,50.0042,36.2358,Kharkiv,Europe/Kiev,2025-01-02,4.3,-1.6,66.1,0.000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
386894,50.6193,26.2513,Rivne,Europe/Kiev,2024-01-02,1.0,0.3,95.6,8.029,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
